# any-reduce-axis — worked example 3: Drop all-zero columns using any(dim, keepdim)

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `any-reduce-axis`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Passing `keepdim=True` to `.any(dim=k)` keeps the reduced axis as a size-1 dimension instead of removing it, so the result still broadcasts cleanly against the original tensor. Combined with boolean indexing or masking, this lets you decide per-column (or per-row) and then act on the full tensor without manual `unsqueeze` calls.

## Worked solution

We have a `(N, M)` float tensor `x`. We want to keep only the columns that contain at least one non-zero entry, returning a possibly-narrower `(N, K)` tensor.

**Step 1 — build the boolean mask.** `x != 0` gives a `(N, M)` bool tensor marking non-zero cells.

**Step 2 — reduce over rows per column.** "Column has any non-zero" ORs down each column, so we collapse axis 0: `(x != 0).any(dim=0)`. That yields a `(M,)` bool vector — one verdict per column. Here we do *not* need keepdim, because we will index columns with this 1-D mask.

**Step 3 — select the surviving columns.** Boolean column selection `x[:, col_keep]` keeps every row but only the columns where `col_keep` is `True`. The result is `(N, K)` where `K` is the number of non-empty columns.

Why `dim=0` and not `dim=1`? We are judging *columns*, and a column spans all rows; ORing over the row axis (axis 0) produces one bool per column — exactly the verdict we index with.

In [ ]:
def drop_allzero_columns(x):
    # x: (N, M) float -> (N, K) keeping columns with any non-zero entry
    col_keep = (x != 0).any(dim=0)  # (M,) bool
    return x[:, col_keep]

t.manual_seed(0)
x = t.tensor([[0.0, 2.0, 0.0, 1.0],
              [0.0, 0.0, 0.0, 3.0]])
result = drop_allzero_columns(x)
print(result, result.shape)